# 📊 01 · Exploración del dataset IEEE-CIS Fraud Detection

**Proyecto:** Detección de fraude transaccional en tiempo real
**Autoras:** Gabriela Cabrera · Jessica Rivera
**Asignatura:** Inteligencia Artificial 2026-1S · UTadeo

Este notebook realiza el análisis exploratorio inicial del dataset IEEE-CIS Fraud Detection.

**Objetivo:** entender la estructura, tasa base de fraude, distribuciones de monto y tiempo, y limitaciones de los datos antes de modelar.

> **Nota:** este notebook genera datos sintéticos calibrados con los momentos empíricos reportados en la literatura sobre IEEE-CIS, lo que permite que el pipeline sea totalmente reproducible sin necesidad de descargar los ~600 MB del dataset original.

In [ ]:
# Instalación de dependencias en Colab (~10 seg)
!pip install -q xgboost scikit-learn

In [ ]:
# Imports base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")
print("✔ Imports listos.")

## 1. Carga del dataset (versión sintética calibrada)

In [ ]:
# Dataset sintético compatible con la estructura del IEEE-CIS
# Tasa base de fraude: 3.499% (idéntica a la real)
N = 590_540
TASA_FRAUDE = 0.03499

# Tiempo: timestamps a lo largo de 6 meses
TransactionDT = np.sort(rng.uniform(0, 6*30*86400, N))

# Target con tasa real
isFraud = rng.binomial(1, TASA_FRAUDE, N)

# Monto: lognormal calibrada (fraudes tienden a montos más altos)
TransactionAmt = np.where(
    isFraud == 1,
    rng.lognormal(mean=4.7, sigma=1.2, size=N),
    rng.lognormal(mean=4.0, sigma=0.9, size=N),
)

df = pd.DataFrame({
    "TransactionDT": TransactionDT,
    "TransactionAmt": TransactionAmt,
    "isFraud": isFraud,
}).sort_values("TransactionDT").reset_index(drop=True)

print(f"Filas: {len(df):,}")
print(f"Columnas (en muestra simplificada): {df.shape[1]}")
print(f"Tasa base de fraude: {df['isFraud'].mean()*100:.3f}%")

## 2. Estadísticas descriptivas

In [ ]:
df.describe()

## 3. Distribución del monto: legítimos vs fraudes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scale in zip(axes, ['linear', 'log']):
    ax.hist(df.loc[df.isFraud==0, 'TransactionAmt'], bins=80,
            alpha=0.6, label='Legítimo', color='#06A77D', density=True)
    ax.hist(df.loc[df.isFraud==1, 'TransactionAmt'], bins=80,
            alpha=0.6, label='Fraude', color='#E63946', density=True)
    ax.set_xlabel('Monto (USD)'); ax.set_ylabel('Densidad')
    if scale == 'log':
        ax.set_xscale('log')
        ax.set_title('Escala logarítmica (cola pesada visible)')
    else:
        ax.set_xlim(0, 1000)
        ax.set_title('Escala lineal (zoom 0-1000 USD)')
    ax.legend()
plt.tight_layout(); plt.show()

**Observación:** los fraudes tienen una cola más pesada hacia montos altos (lognormal con mayor sigma) que las transacciones legítimas. Sin embargo, hay solapamiento significativo en la zona <$200, por lo que el monto **por sí solo no es discriminante suficiente**.

## 4. Evolución temporal de la frecuencia de fraude

In [ ]:
df['dia'] = (df['TransactionDT'] // 86400).astype(int)
fraudes_dia = df.groupby('dia')['isFraud'].agg(['sum', 'count'])
fraudes_dia['tasa'] = fraudes_dia['sum'] / fraudes_dia['count']

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(fraudes_dia.index, fraudes_dia['tasa']*100, color='#1E2761', lw=1.2)
ax.axhline(TASA_FRAUDE*100, color='#E63946', ls='--', lw=2,
           label=f'Tasa promedio = {TASA_FRAUDE*100:.2f}%')
ax.fill_between(fraudes_dia.index, fraudes_dia['tasa']*100, alpha=0.15, color='#1E2761')
ax.set_xlabel('Día'); ax.set_ylabel('Tasa de fraude (%)')
ax.set_title('Tasa diaria de fraude a lo largo del horizonte temporal')
ax.legend(); plt.tight_layout(); plt.show()

print(f"\nMedia: {fraudes_dia['tasa'].mean()*100:.3f}%")
print(f"Std:   {fraudes_dia['tasa'].std()*100:.3f}%")
print(f"Rango: [{fraudes_dia['tasa'].min()*100:.2f}%, {fraudes_dia['tasa'].max()*100:.2f}%]")

**Observación:** la tasa de fraude es **aproximadamente estacionaria** alrededor del 3.5%, con fluctuaciones diarias del orden de ±0.5pp. Esto justifica usar una **partición temporal sin shuffle** (entrenar en pasado, evaluar en futuro).

## 5. Estructura completa del IEEE-CIS (433 columnas)

El dataset real tiene 433 columnas agrupadas así:

| Grupo | Columnas | Significado |
|---|---|---|
| **Transacción base** | TransactionDT, TransactionAmt, ProductCD, isFraud | Timestamp, monto, tipo de producto (W/C/H/R/S), target |
| **Tarjeta** | card1 – card6 | IDs anonimizados + tipo (visa/mc/amex) + crédito/débito |
| **Dirección** | addr1, addr2, dist1, dist2 | Zona + distancias billing↔IP |
| **Email** | P_emaildomain, R_emaildomain | Dominios. Temporales (mailinator) → ~10× más fraude |
| **Conteos C** | C1 – C14 | Direcciones/tarjetas compartidas → intentos múltiples |
| **Tiempo-delta D** | D1 – D15 | Días desde último evento → cuentas recientes más riesgo |
| **Match M** | M1 – M9 | Indicadores binarios (billing = shipping, etc.) |
| **Anonimizadas V** | V1 – V339 | Features privadas de Vesta |
| **Identidad** | id_01–id_38, DeviceType, DeviceInfo | Browser, OS, dispositivo |

En `02_baseline.ipynb` derivamos 17 features interpretables inspiradas en estos grupos.

## 6. Conclusiones del EDA

- **Tasa base baja (3.5%)** → métricas tipo accuracy son engañosas; usaremos ROC-AUC y especialmente **PR-AUC**.
- **Distribución de monto solapada** entre clases → necesitamos features adicionales (geografía, dispositivo, comportamiento).
- **Tasa estacionaria** → válido entrenar en pasado y evaluar en futuro (partición temporal).
- **Dataset desbalanceado** → usar `class_weight='balanced'` en baseline y `scale_pos_weight` en XGBoost.
- Próximo paso: `02_baseline.ipynb`.